# Extraccion de caracteristicas (Espacial)

In [1]:
# --- Celda 1: Setup y Carga ---
import numpy as np
import pandas as pd
import os

FS = 128
WIN_SEC = 2.0
K_MODES = 3  # Tomaremos los primeros 3 modos (baja frecuencia espacial) como el equivalente a la banda alfa

STFT_PATH = './data/tf_representations/stft.npz'
GSP_PATH  = './data/tf_representations/gsp.npz'
TSP_PATH  = './data/tf_representations/tsp.npz'

stft = np.load(STFT_PATH)
gsp  = np.load(GSP_PATH, allow_pickle=True)
tsp  = np.load(TSP_PATH, allow_pickle=True)

stft_times  = stft['times']
stft_labels = stft['labels']

print(f"Reloj Maestro STFT cargado: {len(stft_times)} ventanas.")

Reloj Maestro STFT cargado: 236 ventanas.


In [2]:
# --- Celda 2: Funciones Espaciales ---

def spatial_energy_abs(power, k=K_MODES):
    """Energía absoluta de los modos espaciales más suaves (los K primeros)."""
    # power ya es la energía por modo. Solo sumamos los primeros k.
    return float(np.sum(power[:k]))

def spatial_energy_rel(power, k=K_MODES):
    """Energía de los K modos más suaves respecto a la energía total del grafo."""
    total_energy = np.sum(power)
    if total_energy <= 0:
        return 0.0
    return float(np.sum(power[:k]) / total_energy)

def spatial_entropy(power):
    """
    Entropía de Shannon sobre los modos espaciales.
    Baja entropía = Sincronización fuerte en pocos modos (ej. ojos cerrados).
    Alta entropía = Desincronización, energía esparcida (ej. ojos abiertos).
    """
    total = power.sum()
    if total <= 0:
        return 0.0
    p = power / total
    p = p[p > 0]
    H = -np.sum(p * np.log2(p))
    H_max = np.log2(len(power))
    return float(H / H_max) if H_max > 0 else 0.0

def spatial_cog(power, eigenvalues):
    """Centro de Gravedad Topológico (Promedio ponderado de autovalores)."""
    denom = np.sum(power)
    if denom <= 0:
        return float(np.mean(eigenvalues))
    return float(np.sum(eigenvalues * power) / denom)

In [3]:
# --- Celda 3: Extracción de Bloques GSP y TSP ---

n_samples = gsp['spectrogram'].shape[1] # 14980
gsp_evals = gsp['eigenvalues']
tsp_evals = tsp['eigenvalues']

rows_gsp = []
rows_tsp = []

for j, t_center in enumerate(stft_times):
    idx_start = max(0,              int(np.round((t_center - WIN_SEC / 2) * FS)))
    idx_end   = min(n_samples - 1,  int(np.round((t_center + WIN_SEC / 2) * FS)))
    
    # --- 1. Extracción GSP ---
    segment_gsp_db = gsp['spectrogram'][:, idx_start : idx_end + 1]
    spectrum_gsp_db = segment_gsp_db.mean(axis=1)
    
    # ¡LA CORRECCIÓN VITAL! Convertir de dB a Potencia Lineal
    spectrum_gsp_linear = 10 ** (spectrum_gsp_db / 10)
    
    rows_gsp.append({
        'GSP_spatial_abs': spatial_energy_abs(spectrum_gsp_linear),
        'GSP_spatial_rel': spatial_energy_rel(spectrum_gsp_linear),
        'GSP_entropy':     spatial_entropy(spectrum_gsp_linear),
        'GSP_cog':         spatial_cog(spectrum_gsp_linear, gsp_evals)
    })
    
    # --- 2. Extracción TSP ---
    segment_tsp_db = tsp['spectrogram'][:, idx_start : idx_end + 1]
    spectrum_tsp_db = segment_tsp_db.mean(axis=1)
    
    # Convertir de dB a Potencia Lineal
    spectrum_tsp_linear = 10 ** (spectrum_tsp_db / 10)
    
    rows_tsp.append({
        'TSP_spatial_abs': spatial_energy_abs(spectrum_tsp_linear),
        'TSP_spatial_rel': spatial_energy_rel(spectrum_tsp_linear),
        'TSP_entropy':     spatial_entropy(spectrum_tsp_linear),
        'TSP_cog':         spatial_cog(spectrum_tsp_linear, tsp_evals)
    })

# Crear DataFrames
df_gsp = pd.DataFrame(rows_gsp)
df_gsp['label'] = stft_labels.astype(int)

df_tsp = pd.DataFrame(rows_tsp)
df_tsp['label'] = stft_labels.astype(int)

print(f"GSP — shape: {df_gsp.shape} | Variables: {list(df_gsp.columns)}")
print(f"TSP — shape: {df_tsp.shape} | Variables: {list(df_tsp.columns)}")

GSP — shape: (236, 5) | Variables: ['GSP_spatial_abs', 'GSP_spatial_rel', 'GSP_entropy', 'GSP_cog', 'label']
TSP — shape: (236, 5) | Variables: ['TSP_spatial_abs', 'TSP_spatial_rel', 'TSP_entropy', 'TSP_cog', 'label']


In [4]:
# --- Celda 4: Sincronización final y Guardado ---

# Simular el cálculo de offset que tu compañero usó para excluir bordes
wpd  = np.load('./data/tf_representations/wpd.npz')
wpd_times = wpd['times']
wpd_idx_map = np.array([np.argmin(np.abs(wpd_times - ti)) for ti in stft_times])
offsets = np.abs(wpd_times[wpd_idx_map] - stft_times)

# Máscara de ventanas válidas (elimina las 5 desalineadas)
valid_mask = offsets <= 0.1

df_gsp_final = df_gsp.loc[valid_mask].reset_index(drop=True)
df_tsp_final = df_tsp.loc[valid_mask].reset_index(drop=True)

print(f"Shape final tras limpiar bordes:")
print(f"  GSP: {df_gsp_final.shape} (231, 5)")
print(f"  TSP: {df_tsp_final.shape} (231, 5)")

# Guardado
output_dir = './data/features/'
os.makedirs(output_dir, exist_ok=True)

df_gsp_final.to_csv(output_dir + 'features_gsp.csv', index=False)
df_tsp_final.to_csv(output_dir + 'features_tsp.csv', index=False)

print("\nArchivos guardados correctamente en ./data/features/")

Shape final tras limpiar bordes:
  GSP: (231, 5) (231, 5)
  TSP: (231, 5) (231, 5)

Archivos guardados correctamente en ./data/features/
